# AeroClean — YOLO11n Training Notebook

**Purpose:** Train a YOLO11n model to detect `clean_board` and `dirty_board` classes,  
then export to NCNN format for deployment on a Raspberry Pi 5.

**Runtime:** Set runtime to **GPU → T4** before running  
*(Runtime → Change runtime type → T4 GPU)*

---

## Workflow overview
1. Mount Google Drive (stores your dataset and checkpoints)
2. Install dependencies
3. Verify GPU
4. Prepare dataset (upload from Pi or Roboflow)
5. Train YOLO11n
6. Evaluate — view mAP, loss curves, confusion matrix
7. Export to NCNN
8. Download weights for deployment on the Pi

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# We'll store everything under this folder in your Drive:
DRIVE_ROOT = '/content/drive/MyDrive/AeroClean'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics --quiet
import ultralytics
ultralytics.checks()   # prints version + GPU info

## Step 3 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Training will be slow. Change runtime to GPU.')

## Step 4 — Upload your dataset

### Option A — Upload a zip from your computer
Run the cell below, click **Choose Files**, and select the zip you created  
with `prepare_dataset.py` on the Pi.

### Option B — Use Roboflow
If you labeled images in Roboflow instead of Label Studio, paste your  
Roboflow export snippet in the **Roboflow cell** below.

The dataset must be in YOLO format:
```
AeroClean/
  images/
    train/  val/  test/
  labels/
    train/  val/  test/
  train/dataset.yaml
```

In [ ]:
# ── Option A: Upload zip ──────────────────────────────────────────────────
from google.colab import files
import zipfile, shutil

print('Select your dataset zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
os.makedirs(DATASET_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

print(f'Dataset extracted to {DATASET_DIR}')
!ls {DATASET_DIR}

In [ ]:
# ── Option B: Roboflow export (skip if you used Option A) ─────────────────
# !pip install roboflow --quiet
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('aeroclean')
# dataset = project.version(1).download('yolov8')   # yolov8 format works for YOLO11
# DATASET_DIR = dataset.location
# print(f'Dataset at {DATASET_DIR}')

## Step 5 — Set dataset.yaml path

In [ ]:
import yaml

# Path to dataset.yaml (adjust if your zip has a different structure)
YAML_PATH = os.path.join(DATASET_DIR, 'train', 'dataset.yaml')

# Update the 'path' key inside dataset.yaml to point to DATASET_DIR
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATASET_DIR

with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f)

print('dataset.yaml updated:')
print(yaml.dump(cfg))

## Step 6 — Train YOLO11n

**Key hyperparameters:**

| Parameter | Value | Notes |
|---|---|---|
| `epochs` | 100 | Increase to 150–200 if mAP plateaus |
| `imgsz` | 640 | Standard for YOLO11n |
| `batch` | 16 | T4 GPU fits 16 comfortably |
| `patience` | 20 | Early stop if no improvement for 20 epochs |
| `augment` | True (default) | Mosaic, flip, HSV jitter |

Training saves checkpoints to `DRIVE_ROOT/runs/` automatically.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # downloads pretrained weights (~2.6MB)

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,                              # GPU 0
    project=os.path.join(DRIVE_ROOT, 'runs'),
    name='aeroclean_yolo11n',
    exist_ok=True,
)

print('Training complete!')
print('Best weights:', results.save_dir)

## Step 7 — Evaluate the model

In [ ]:
from IPython.display import Image, display
import glob

run_dir = os.path.join(DRIVE_ROOT, 'runs', 'aeroclean_yolo11n')

# Validation metrics (mAP50, mAP50-95, precision, recall)
val_results = model.val(data=YAML_PATH, split='val')
print(f'mAP50:    {val_results.box.map50:.4f}')
print(f'mAP50-95: {val_results.box.map:.4f}')

# Training curves
for img in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(path))

## Step 8 — Export to NCNN

NCNN is the fastest inference format on the Raspberry Pi 5 ARM CPU.  
The export produces a folder containing `*.param` and `*.bin` files.

In [ ]:
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
export_model = YOLO(best_weights)

export_path = export_model.export(format='ncnn')
print(f'NCNN model saved to: {export_path}')

## Step 9 — Download weights to your computer

After downloading, copy the `best_ncnn_model/` folder to  
the Pi at `AeroClean/weights/best_ncnn_model/`.

In [ ]:
import shutil
from google.colab import files

ncnn_dir = export_path   # path returned by model.export()
zip_out = '/content/best_ncnn_model.zip'
shutil.make_archive('/content/best_ncnn_model', 'zip', ncnn_dir)

print('Downloading best_ncnn_model.zip ...')
files.download(zip_out)

## Step 10 — Deploy to the Raspberry Pi

```bash
# On your laptop — copy the weights to the Pi:
scp best_ncnn_model.zip pi@<PI_IP>:~/AeroClean/weights/

# On the Pi — unzip:
cd ~/AeroClean/weights
unzip best_ncnn_model.zip -d best_ncnn_model

# Run inference:
cd ~/AeroClean
python main.py --model yolo
```

The `config.json` key `yolo_weights` already points to `weights/best_ncnn_model` by default.